# CASCADE-ER v2 — Preprocessing & Wide Blocking Runner (Google Colab)

This notebook executes both **Stage 0 (Preprocessing & Normalization)** and **Stage 1 (Wide Blocking)** sequentially on Google Colab with GPU acceleration and Google Drive synchronization.

```text
Raw Datasets (data/raw/train, data/raw/test)
                 │
                 ▼
STAGE 0: PREPROCESSING & NORMALIZATION
                 │
                 ▼
Preprocessed TSVs (data/processed/train, data/processed/test)
                 │
                 ▼
STAGE 1: WIDE BLOCKING (6 Channels Union)
                 │
                 ▼
Candidate Union (data/candidates/stage1_candidate_union.tsv)
```

--- 
## 1. Clone / Update Repository & Install Dependencies

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/Blackdevil-com/Wired-Mortus---Amazon-ML.git"
REPO_NAME = "Wired-Mortus---Amazon-ML"

# Clone if not present, then enter directory and pull latest code
if not Path("src/blocking").exists():
    if not Path(REPO_NAME).exists():
        !git clone {REPO_URL}
    %cd {REPO_NAME}

!git fetch origin main
!git reset --hard origin/main

print(f"\nCurrent working directory: {Path.cwd()}")
!pip install -r requirements-colab.txt

--- 
## 2. Mount Google Drive & Link Raw Datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Base Google Drive directories
DRIVE_BASE = Path('/content/drive/MyDrive/entity-resolution')
DRIVE_RAW_TRAIN = DRIVE_BASE / 'data/raw/train'
DRIVE_RAW_TEST = DRIVE_BASE / 'data/raw/test'
DRIVE_PROC_TRAIN = DRIVE_BASE / 'data/processed/train'
DRIVE_PROC_TEST = DRIVE_BASE / 'data/processed/test'
DRIVE_CANDIDATES = DRIVE_BASE / 'data/candidates'

# Create local & Drive directories
!mkdir -p data/raw/train data/raw/test data/processed/train data/processed/test data/candidates data/cache
DRIVE_PROC_TRAIN.mkdir(parents=True, exist_ok=True)
DRIVE_PROC_TEST.mkdir(parents=True, exist_ok=True)
DRIVE_CANDIDATES.mkdir(parents=True, exist_ok=True)

# Copy raw datasets from Drive if available
if DRIVE_RAW_TRAIN.exists():
    print(f"Copying raw train datasets from {DRIVE_RAW_TRAIN}...")
    !cp -n "{DRIVE_RAW_TRAIN}"/*.tsv data/raw/train/ 2>/dev/null || true

if DRIVE_RAW_TEST.exists():
    print(f"Copying raw test datasets from {DRIVE_RAW_TEST}...")
    !cp -n "{DRIVE_RAW_TEST}"/*.tsv data/raw/test/ 2>/dev/null || true

# Copy preprocessed datasets from Drive if available (skips re-running preprocessing!)
if DRIVE_PROC_TRAIN.exists():
    print(f"Copying preprocessed train datasets from {DRIVE_PROC_TRAIN}...")
    !cp -n "{DRIVE_PROC_TRAIN}"/*.tsv data/processed/train/ 2>/dev/null || true

if DRIVE_PROC_TEST.exists():
    print(f"Copying preprocessed test datasets from {DRIVE_PROC_TEST}...")
    !cp -n "{DRIVE_PROC_TEST}"/*.tsv data/processed/test/ 2>/dev/null || true

print("\n--- Local Datasets Detected ---")
print("Raw Train (data/raw/train):")
for f in sorted(Path("data/raw/train").glob("*.tsv")):
    print(f"  - {f.name} ({f.stat().st_size / (1024 * 1024):.2f} MB)")

print("\nProcessed Train (data/processed/train):")
proc_train = list(sorted(Path("data/processed/train").glob("*.tsv")))
if proc_train:
    for f in proc_train:
        print(f"  - {f.name} ({f.stat().st_size / (1024 * 1024):.2f} MB)")
else:
    print("  (Empty - need to copy from Drive or run Step 4 Preprocessing)")


--- 
## 3. Run Pytest Suite

In [ ]:
!pytest tests/ -v

--- 
## 4. STAGE 0: Execute Preprocessing & Normalization
Normalizes business names, addresses, countries, extracts tokens and row numbers for both training and testing sets.

In [ ]:
# Preprocess Training data
!python run_preprocessing.py --split train --raw-dir data/raw/train --processed-dir data/processed/train

# Preprocess Testing data
!python run_preprocessing.py --split test --raw-dir data/raw/test --processed-dir data/processed/test

# Upload preprocessed TSVs to Google Drive
print("\nUploading preprocessed TSVs to Google Drive...")
!cp -v data/processed/train/*.tsv "{DRIVE_PROC_TRAIN}/" 2>/dev/null || true
!cp -v data/processed/test/*.tsv "{DRIVE_PROC_TEST}/" 2>/dev/null || true

--- 
## 5. STAGE 1: Execute Wide Blocking (High-Recall Candidate Union)
Runs all 6 blocking channels (exact name, core name, house+street, postal, char TF-IDF, dense embeddings/FAISS) and generates `data/candidates/stage1_candidate_union.tsv`.

In [ ]:
# Run Stage 1 Wide Blocking on Training data
!python run_stage1_blocking.py \
    --split train \
    --s1 data/processed/train/train_source1_preprocessed.tsv \
    --s2 data/processed/train/train_source2_preprocessed.tsv \
    --s3 data/processed/train/train_source3_preprocessed.tsv \
    --ground-truth data/raw/train/train_ground_truth.tsv \
    --tfidf-top-k 30 \
    --embedding-top-k 20 \
    --output data/candidates/stage1_candidate_union.tsv

# Upload Candidate Union to Google Drive
print("\nUploading Stage 1 candidate union to Google Drive...")
!cp -v data/candidates/stage1_candidate_union.tsv "{DRIVE_CANDIDATES}/"

--- 
## 6. Inspect Candidate Union Output

In [ ]:
import pandas as pd

cand_file = Path("data/candidates/stage1_candidate_union.tsv")
if cand_file.exists():
    df = pd.read_csv(cand_file, sep="\t", nrows=10, dtype=str, keep_default_na=False)
    print(f"Candidate Union Preview (Total columns: {len(df.columns)}):\n")
    display(df)